# 05 — LoRA SFT of Gemma on SAC Trajectories · Google Colab (Unsloth)

**Phase 2 / Month 2** · MSc Thesis — ECLIPSE project  
Supervisor: Dr. Panagiotis Kasnesis | Student: Antonios Bastoulis

---

Companion to notebook **04** (local). Pipeline:

1. Pull the SAC distillation JSONL produced by 04.
2. Load Gemma-3n E4B via **Unsloth** (`FastModel`) — 4-bit + LoRA, ~2× faster
   training and ~50% less VRAM than the vanilla HF stack.
3. Run SFT with TRL's `SFTTrainer` on Unsloth's optimized model.
4. Evaluate the fine-tuned SLM in CityLearn (6-building single-agent rollout)
   against RBC and the no-control baseline using the canonical `src.eval` KPIs.

Reference: https://unsloth.ai/docs/models/gemma-4/train

Set runtime to **T4 GPU** (free tier) — Unsloth fits Gemma-3n E4B + LoRA comfortably.

## § 0 — Runtime & Config
> **Edit this cell only.** Nothing else needs changing.

In [ ]:
import os, sys, subprocess, time, warnings, json, random
import numpy as np

# ── Repo + dataset selection ──────────────────────────────────────────────
GITHUB_REPO    = "https://github.com/antonisbast/eclipse-thesis.git"  # adjust if needed
REPO_DIR       = "/content/eclipse-thesis"
DATASET_GLOB   = "notebooks/artifacts/sft_datasets/sac_*.jsonl"  # picks newest match

# ── Model selection ───────────────────────────────────────────────────────
# Unsloth-optimized Gemma 3n E4B — same architecture as notebook 03's
# google/gemma-4-E4B-it. Use the unsloth/* variant to get the fast kernels.
MODEL_ID:       str  = "unsloth/gemma-3n-E4B-it"
LOAD_IN_4BIT:   bool = True
MAX_NEW_TOKENS: int  = 200

# ── LoRA / SFT hyperparameters ────────────────────────────────────────────
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.0    # Unsloth recommends 0 for fastest path
EPOCHS         = 1
BATCH_SIZE     = 2
GRAD_ACCUM     = 8      # effective batch 16
LEARNING_RATE  = 2e-4
MAX_SEQ_LEN    = 1024

# ── Evaluation slice ──────────────────────────────────────────────────────
EVAL_START     = 0
EVAL_LEN       = 168              # 1-week eval (smoke test); raise to 8760 for full year

# ── Misc ──────────────────────────────────────────────────────────────────
HF_TOKEN: str = ""
SEED          = 42

try:
    import torch
    if torch.cuda.is_available():
        _gpu  = torch.cuda.get_device_name(0)
        _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✓ GPU: {_gpu}  ({_vram:.1f} GB VRAM)")
    else:
        print("⚠  No GPU — set Runtime → Change runtime type → T4 GPU")
except ImportError:
    print("torch not yet installed — will be in § 1")

## § 1 — Install Dependencies

In [ ]:
# CityLearn (for evaluation rollout)
!pip install -q numpy gymnasium doe-xstock nrel-pysam
!pip install -q CityLearn --no-deps
print("✓ CityLearn installed")

In [ ]:
# Unsloth installs its own pinned versions of transformers, peft, trl,
# bitsandbytes, accelerate. Per Unsloth's Colab quick-start, install
# unsloth + unsloth_zoo first, then let it pull compatible deps.
!pip install -q --upgrade pip
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps git+https://github.com/unslothai/unsloth-zoo.git
!pip install -q --upgrade transformers peft trl accelerate bitsandbytes datasets sentencepiece
# triton + xformers are pre-installed on Colab; ensure recent enough
!pip install -q --upgrade triton
print("✓ Unsloth + SFT stack installed")

## § 2 — Clone Repository (pulls `src/sft.py` and the JSONL dataset)

In [ ]:
if not os.path.exists(REPO_DIR):
    res = subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR],
                         capture_output=True, text=True)
    print(res.stdout or res.stderr)
else:
    res = subprocess.run(["git", "-C", REPO_DIR, "pull"],
                         capture_output=True, text=True)
    print("Repo present —", res.stdout.strip() or "up to date")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

## § 3 — Load Distillation Dataset

In [ ]:
import glob
from pathlib import Path

candidates = sorted(glob.glob(os.path.join(REPO_DIR, DATASET_GLOB)))
assert candidates, f"No JSONL matching {DATASET_GLOB} in repo. Run notebook 04 + push first."
JSONL_PATH = candidates[-1]
print(f"Using dataset: {JSONL_PATH}")

raw_rows = [json.loads(l) for l in Path(JSONL_PATH).read_text().strip().split("\n")]
print(f"Examples: {len(raw_rows)}")
print("\nFirst row preview:")
print("  prompt :", raw_rows[0]["prompt"][:200].replace("\n", " | "))
print("  response:", raw_rows[0]["response"][:200].replace("\n", " | "))

## § 4 — Load Gemma + Attach LoRA (Unsloth `FastModel`)

`FastModel.from_pretrained` returns the model **and** tokenizer with Unsloth's
kernels patched in. `FastModel.get_peft_model` attaches LoRA adapters in
Unsloth-optimized form (no `prepare_model_for_kbit_training` needed).

In [ ]:
from unsloth import FastModel
import torch

_t0 = time.time()
model, tokenizer = FastModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ_LEN,
    load_in_4bit    = LOAD_IN_4BIT,
    full_finetuning = False,
    token           = HF_TOKEN or None,
)
print(f"Base model loaded in {time.time()-_t0:.1f}s")

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,   # text-only distillation
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    random_state   = SEED,
)
model.print_trainable_parameters()

## § 5 — Build HF Dataset with Gemma Chat Template

Gemma rejects the `system` role, so the SFT prompt is merged into the user
turn (same workaround as notebook 03's `LocalHFProvider`).

In [ ]:
from datasets import Dataset

from src.sft import make_sft_prompt

N_BUILDINGS   = 6
SYSTEM_PROMPT = make_sft_prompt(N_BUILDINGS)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_is_gemma = "gemma" in MODEL_ID.lower()

def to_chat(row):
    if _is_gemma:
        msgs = [
            {"role": "user",      "content": f"{SYSTEM_PROMPT}\n\n{row['prompt']}"},
            {"role": "assistant", "content": row["response"]},
        ]
    else:
        msgs = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": row["prompt"]},
            {"role": "assistant", "content": row["response"]},
        ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {"text": text}

ds = Dataset.from_list(raw_rows).map(to_chat, remove_columns=["prompt", "response"])
print(f"Dataset built: {len(ds)} examples  |  fields: {ds.column_names}")
print("\nSample formatted text (truncated):")
print(ds[0]["text"][:500])
print("...")

## § 6 — Train (TRL `SFTTrainer` on Unsloth model)

Unsloth's patched model + tokenizer plug straight into TRL's `SFTTrainer`.
We train causal-LM loss on the full sequence (prompt + completion) — fine
for behaviour-cloning distillation. For loss-on-completion-only, swap to
`DataCollatorForCompletionOnlyLM`.

In [ ]:
from trl import SFTTrainer, SFTConfig

OUT_DIR = f"/content/sft_out_{time.strftime('%Y%m%d_%H%M%S')}"

sft_cfg = SFTConfig(
    output_dir                  = OUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    max_length                  = MAX_SEQ_LEN,
    logging_steps               = 25,
    save_strategy               = "no",
    bf16                        = False,
    fp16                        = True,
    optim                       = "adamw_8bit",   # Unsloth-recommended on T4
    warmup_ratio                = 0.03,
    lr_scheduler_type           = "linear",
    report_to                   = "none",
    seed                        = SEED,
    dataset_text_field          = "text",
)

trainer = SFTTrainer(
    model           = model,
    args            = sft_cfg,
    train_dataset   = ds,
    processing_class= tokenizer,
)

_t0 = time.time()
trainer.train()
print(f"\nSFT done in {time.time()-_t0:.1f}s")

ADAPTER_DIR = f"{OUT_DIR}/lora_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved: {ADAPTER_DIR}")

## § 7 — Build Inference Provider (Unsloth fast inference)

`FastModel.for_inference(model)` enables Unsloth's 2× faster inference path
(disables training-only autograd hooks). Same `.complete()` / `.step()` API
as notebook 03's `LocalHFProvider`, so the rollout helper below is unchanged.

In [ ]:
import logging, re
from src.sft import parse_actions, make_sft_prompt

_logger = logging.getLogger("sft_slm")
_ACTION_RE = re.compile(r"<action\s+building\s*=\s*\d+\s*>", re.IGNORECASE)

FastModel.for_inference(trainer.model)  # 2× faster inference


class SFTProvider:
    """Inference provider wrapping the LoRA-fine-tuned Unsloth model."""

    def __init__(self, model, tokenizer, max_new_tokens: int = 200):
        self.model         = model.eval()
        self.tokenizer     = tokenizer
        self.max_new_tokens= max_new_tokens
        self.label         = f"sft:{MODEL_ID.split('/')[-1]}"
        self._is_gemma     = "gemma" in MODEL_ID.lower()
        self._device       = next(model.parameters()).device

    def complete(self, system: str, user: str, max_tokens=None) -> str:
        max_new = max_tokens or self.max_new_tokens
        if self._is_gemma:
            msgs = [{"role": "user", "content": f"{system}\n\n{user}"}]
        else:
            msgs = [{"role": "system", "content": system},
                    {"role": "user",   "content": user}]
        enc = self.tokenizer.apply_chat_template(
            msgs, tokenize=True, add_generation_prompt=True,
            return_tensors="pt", return_dict=True,
        ).to(self._device)
        with torch.no_grad():
            out = self.model.generate(
                **enc,
                max_new_tokens=max_new,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
            )
        new_tokens = out[0][enc["input_ids"].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    def step(self, state_text: str, system=None, n_buildings: int = 6,
             max_retries: int = 2, **kw):
        sys_p = system or make_sft_prompt(n_buildings)
        last  = ""
        for _ in range(max_retries):
            last = self.complete(sys_p, f"STATE:\n{state_text}")
            if _ACTION_RE.search(last):
                return parse_actions(last, n_buildings), last, False
        return [0.0]*n_buildings, last, True


slm = SFTProvider(trainer.model, tokenizer, max_new_tokens=MAX_NEW_TOKENS)
print(f"Provider ready: {slm.label}")

## § 8 — Evaluation Environment Factory (Colab)

In [ ]:
import citylearn
from citylearn.citylearn import CityLearnEnv
from citylearn.agents.rbc import BasicBatteryRBC
from src.env import (
    snapshot_state, MERLINReward, OBSERVATIONS_LLM, ACTIVE_ACTIONS, BUILDINGS,
)
from src.eval import evaluate, comparison_table
from src.sft  import render_state

warnings.filterwarnings("ignore")
np.random.seed(SEED); random.seed(SEED)

def make_colab_env(start=0, end=8758, buildings=None) -> CityLearnEnv:
    env = CityLearnEnv(
        schema="citylearn_challenge_2022_phase_all",
        buildings=buildings if buildings is not None else BUILDINGS,
        central_agent=False,
        active_actions=ACTIVE_ACTIONS,
        active_observations=OBSERVATIONS_LLM,
        random_seed=SEED,
        simulation_start_time_step=start,
        simulation_end_time_step=end,
    )
    env.reward_function = MERLINReward(env.get_metadata())
    return env

print(f"CityLearn {citylearn.__version__}")

## § 9 — Rollout Helpers

In [ ]:
def run_slm_policy(name: str, provider, start=EVAL_START, length=EVAL_LEN,
                   n_buildings: int = N_BUILDINGS, verbose_every: int = 24):
    env = make_colab_env(start=start, end=start+length-1)
    env.reset()
    sys_p = make_sft_prompt(n_buildings)
    done, t, t0, n_fb = False, 0, time.time(), 0
    while not done:
        snap = snapshot_state(env)
        state_text = render_state(snap)
        acts, raw, fb = provider.step(state_text, system=sys_p, n_buildings=n_buildings)
        n_fb += int(fb)
        _, _, term, trunc, _ = env.step([[float(a)] for a in acts])
        done = bool(term or trunc)
        if verbose_every and (t % verbose_every == 0):
            elapsed = time.time() - t0
            print(f"  t={t:4d} | {elapsed:.0f}s | fallbacks={n_fb}")
        t += 1
    print(f"[{name}] {t} steps in {time.time()-t0:.1f}s | fallbacks={n_fb}/{t}")
    return env


def run_rbc(start=EVAL_START, length=EVAL_LEN):
    env = make_colab_env(start=start, end=start+length-1)
    BasicBatteryRBC(env=env).learn(episodes=1)
    return env


def run_noop(start=EVAL_START, length=EVAL_LEN):
    env = make_colab_env(start=start, end=start+length-1)
    env.reset()
    n_b = len(env.buildings)
    done = False
    while not done:
        _, _, term, trunc, _ = env.step([[0.0] for _ in range(n_b)])
        done = bool(term or trunc)
    return env

## § 10 — Run Baselines + SFT-SLM

In [ ]:
print("── No-control baseline ──")
env_noop = run_noop()

print("\n── RBC ──")
env_rbc  = run_rbc()

print(f"\n── SFT SLM ({slm.label}) ──")
env_slm  = run_slm_policy("sft_slm", slm)

## § 11 — Evaluation

In [ ]:
res_noop = evaluate(env_noop, label="No-Control")
res_rbc  = evaluate(env_rbc,  label="RBC")
res_slm  = evaluate(env_slm,  label=slm.label)

challenge_df, zne_df = comparison_table([res_noop, res_rbc, res_slm])
print("Challenge scores (lower is better; 1.0 = no-control baseline):")
display(challenge_df)
print("\nZNE / self-consumption:")
display(zne_df[["ZNE ratio (solar / import)", "self-consumption ratio"]])

## § 12 — Persist Adapter to Drive (optional)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r {ADAPTER_DIR} /content/drive/MyDrive/eclipse-thesis/sft_adapters/
print(f"Adapter at: {ADAPTER_DIR}")
print("Uncomment the cell above to persist to Google Drive.")